# 18D — Merge repaired-slot research into sanitized Batch 1–3

Run only after the three `V5_BATCH_01_03_RERESEARCH_*` files have been placed under:

`research/ml/artifacts/v5_event_collection_repaired/reresearch_01_03/`

This notebook restores Batch 1–3 after identity repair.

It does not pair events or generate astrology.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib,json
import pandas as pd

def root():
    p=Path.cwd().resolve()
    for x in [p]+list(p.parents):
        if (x/"saju_engine.py").exists(): return x
    raise FileNotFoundError("repo root")
def sha(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""):h.update(c)
    return h.hexdigest()

ROOT=root()
BASE=ROOT/"research/ml/artifacts/v5_event_collection_repaired"
SAN=BASE/"results_sanitized"
RR=BASE/"reresearch_01_03"
OUT=BASE/"results_post_repair"
POST=ROOT/"research/ml/artifacts/v5_post_repair_validation"
CORP=ROOT/"research/ml_corpus/v5_ground_truth"
OUT.mkdir(parents=True,exist_ok=True)

DEC=POST/"V5_POST_REPAIR_VALIDATION_DECISION.json"
FIX=POST/"V5_BATCH_01_03_RERESEARCH_SUBJECTS.csv"
RE=RR/"V5_BATCH_01_03_RERESEARCH_EVENTS.csv"
RA=RR/"V5_BATCH_01_03_RERESEARCH_SWEEP_AUDIT.csv"
RM=RR/"V5_BATCH_01_03_RERESEARCH_MANIFEST.json"
PROTO=CORP/"V5_BATCH_01_03_POST_REPAIR_MERGE_PROTOCOL.json"

for p in [DEC,FIX,RE,RA,RM,PROTO]:
    if not p.exists(): raise FileNotFoundError(p)

dec=json.load(open(DEC,encoding="utf-8"))
rm=json.load(open(RM,encoding="utf-8"))
proto=json.load(open(PROTO,encoding="utf-8"))
fixed=pd.read_csv(FIX)
re=pd.read_csv(RE)
ra=pd.read_csv(RA)

assert dec["status"]=="V5_POST_REPAIR_VALIDATION_PASS_RERESEARCH_6_REQUIRED"
assert rm["status"]=="V5_BATCH_01_03_REPAIRED_SLOTS_RERESEARCH_COMPLETE_READY_FOR_MERGE_VALIDATION"
assert proto["status"]=="PREDECLARED_AFTER_FIXED_RERESEARCH_BEFORE_MERGE"
assert sha(FIX)==dec["reresearch_subjects_sha256"]
assert sha(RE)==rm["events_sha256"]
assert sha(RA)==rm["audit_sha256"]
assert set(re.subject_id)==set(fixed.subject_id)
assert set(ra.subject_id)==set(fixed.subject_id)

print("18D PREFLIGHT PASS")


18D PREFLIGHT PASS


In [2]:

summ=[]
for b, expected_n in [(1,21),(2,21),(3,20)]:
    src=SAN/f"batch_{b:02d}"
    evs=src/f"V5_DEV_EVENT_BATCH_{b:02d}_EVENTS_SANITIZED.csv"
    aus=src/f"V5_DEV_EVENT_BATCH_{b:02d}_SUBJECT_SWEEP_AUDIT_SANITIZED.csv"
    sm=src/f"V5_DEV_EVENT_BATCH_{b:02d}_SANITIZATION_MANIFEST.json"
    for p in [evs,aus,sm]:
        if not p.exists(): raise FileNotFoundError(p)

    ev=pd.read_csv(evs); au=pd.read_csv(aus)
    newev=re[re.batch_id==b].copy()
    newau=ra[ra.batch_id==b].copy()

    # fixed subject IDs must not remain in sanitized data.
    ids=set(fixed.loc[fixed.batch_id==b,"subject_id"].astype(str))
    assert set(ev.subject_id.astype(str)).isdisjoint(ids)
    assert set(au.subject_id.astype(str)).isdisjoint(ids)
    assert set(newev.subject_id.astype(str))==ids
    assert set(newau.subject_id.astype(str))==ids

    mev=pd.concat([ev,newev],ignore_index=True,sort=False)
    mau=pd.concat([au,newau],ignore_index=True,sort=False)

    assert mau.subject_id.nunique()==expected_n
    assert len(mau)==expected_n
    assert mev.subject_id.nunique()==expected_n
    assert set(mev.subject_id)==set(mau.subject_id)

    dst=OUT/f"batch_{b:02d}"
    dst.mkdir(parents=True,exist_ok=True)
    ep=dst/f"V5_DEV_EVENT_BATCH_{b:02d}_EVENTS_POST_REPAIR.csv"
    ap=dst/f"V5_DEV_EVENT_BATCH_{b:02d}_SUBJECT_SWEEP_AUDIT_POST_REPAIR.csv"
    mev.to_csv(ep,index=False); mau.to_csv(ap,index=False)

    m={
      "version":f"V5_DEV_EVENT_BATCH_{b:02d}_POST_REPAIR_V1",
      "status":"POST_REPAIR_BATCH_RESEARCH_COMPLETE_NOT_EVENT_FREEZE",
      "batch_id":b,
      "n_subjects":expected_n,
      "event_rows":len(mev),
      "eligible_rows":int((~mev.exclude.astype(bool)).sum()),
      "events_sha256":sha(ep),
      "audit_sha256":sha(ap),
      "repaired_subject_ids":sorted(ids),
      "membership_changed":False,
      "pairing_performed":False,
      "astrology_generated":False
    }
    mp=dst/f"V5_DEV_EVENT_BATCH_{b:02d}_POST_REPAIR_MANIFEST.json"
    json.dump(m,open(mp,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    summ.append(m)

summary=pd.DataFrame(summ)
sp=OUT/"V5_BATCH_01_03_POST_REPAIR_SUMMARY.csv"
summary.to_csv(sp,index=False)

decision={
 "version":"V5_BATCH_01_03_POST_REPAIR_DECISION_V1",
 "created_at":datetime.now().isoformat(timespec="seconds"),
 "status":"V5_BATCH_01_03_POST_REPAIR_COMPLETE_BATCH_04_MAY_RESUME",
 "batch_subject_counts":{"1":21,"2":21,"3":20},
 "repaired_slots_merged":len(fixed),
 "summary_sha256":sha(sp),
 "rules":{
   "pairing_performed":False,
   "chronology_used":False,
   "astrology_used":False,
   "control_used":False
 },
 "batch_04_may_resume":True,
 "next_rule":"Research repaired Batch 04 subject list only; continue event collection through Batch 8 before event freeze."
}
dp=OUT/"V5_BATCH_01_03_POST_REPAIR_DECISION.json"
json.dump(decision,open(dp,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
print(json.dumps(decision,ensure_ascii=False,indent=2))


{
  "version": "V5_BATCH_01_03_POST_REPAIR_DECISION_V1",
  "created_at": "2026-08-17T03:46:36",
  "status": "V5_BATCH_01_03_POST_REPAIR_COMPLETE_BATCH_04_MAY_RESUME",
  "batch_subject_counts": {
    "1": 21,
    "2": 21,
    "3": 20
  },
  "repaired_slots_merged": 6,
  "summary_sha256": "c9720ee2115fb49aa75f5938e8dbc09de098abe43899f7dcfa2c2c326419f87a",
  "rules": {
    "pairing_performed": false,
    "chronology_used": false,
    "astrology_used": false,
    "control_used": false
  },
  "batch_04_may_resume": true,
  "next_rule": "Research repaired Batch 04 subject list only; continue event collection through Batch 8 before event freeze."
}


## Send back after Run All

Send:

```text
V5_BATCH_01_03_POST_REPAIR_DECISION.json
V5_BATCH_01_03_POST_REPAIR_SUMMARY.csv
```

If status is:

`V5_BATCH_01_03_POST_REPAIR_COMPLETE_BATCH_04_MAY_RESUME`

then send the repaired Batch 04 subject file next.
